# Full Metrics Computation for Synthetic Scatterplots

Computes the **full set of metrics** for all 114,176 synthetic scatterplot cases.
Core methods are identical to `metrics_computation.ipynb`; this notebook adds
covariance, distribution tests, equal-count bins, GAM, and y-SD normalisation.

**Input:** `generated_scatterplot_data/cases.csv` + `scatter_points.npz`

**Output:** `generated_scatterplot_data/full/metrics_full.parquet`

| Group | Metrics | Count |
|---|---|---|
| Correlation | Pearson r, Spearman ρ, covariance | 3 |
| Distance | distance covariance, distance correlation | 2 |
| X coverage | KS from uniform, x bin count CV | 2 |
| Distribution | KS + Wasserstein × equal-width / equal-count | 4 |
| Slopes | endpoint & polyfit × raw/std × overall/early/mid/late + strength | 20 |
| MINE | MIC, MAS, MEV, MCN, MIC − r² | 5 |
| Bin (equal-width) | amplitude, η², buffer widths, n_valid_bins | 7 |
| Bin (equal-count) | amplitude, η², buffer widths, n_valid_bins | 7 |
| LOWESS | residual SD, amplitude, R², sign changes, overall slope | 5 |
| GAM | residual SD, amplitude, R², sign changes, overall slope | 5 |
| y-SD normalised | y_sd + 16 normalised ratios | 17 |
| Misc | n_valid | 1 |
| **Total** | | **78** |

In [11]:
from __future__ import annotations

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, ks_2samp, wasserstein_distance
from statsmodels.nonparametric.smoothers_lowess import lowess

try:
    from minepy import MINE
    HAS_MINEPY = True
except ImportError:
    HAS_MINEPY = False
    print("minepy not installed → MIC / MAS / MEV / MCN will be NaN.")

try:
    from pygam import LinearGAM, s as gam_s
    HAS_PYGAM = True
except ImportError:
    HAS_PYGAM = False
    print("pygam not installed → GAM metrics will be NaN.")

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings("ignore", category=np.RankWarning)

DATA_DIR = Path("generated_scatterplot_data")

## Load Generated Data

In [12]:
cases_df = pd.read_csv(DATA_DIR / "cases.csv", low_memory=False)
data = np.load(DATA_DIR / "scatter_points.npz")
x_all = data["x"]
y_all = data["y"]
n_cases, n_points = x_all.shape
print(f"Loaded {n_cases:,} cases × {n_points} points")
print(f"Family distribution: Null={int((cases_df.family_id == 'Null').sum())}, Signal={int((cases_df.family_id != 'Null').sum())}")

Loaded 114,176 cases × 500 points
Family distribution: Null=128, Signal=114048


## Metric Functions

### Phase 1 — Vectorised (all cases at once)

In [13]:
def vectorised_pearson(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Row-wise Pearson r.  x, y: (n_cases, n_points)."""
    xc = x - x.mean(axis=1, keepdims=True)
    yc = y - y.mean(axis=1, keepdims=True)
    num = (xc * yc).sum(axis=1)
    den = np.sqrt((xc ** 2).sum(axis=1) * (yc ** 2).sum(axis=1))
    return np.where(den > 0, num / den, np.nan)


def _rank_rows(arr: np.ndarray) -> np.ndarray:
    n_cases, n_points = arr.shape
    ranks = np.empty(arr.shape, dtype=np.float64)
    order = arr.argsort(axis=1)
    rows = np.arange(n_cases)[:, None]
    ranks[rows, order] = np.arange(1, n_points + 1, dtype=np.float64)
    return ranks


def vectorised_spearman(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    return vectorised_pearson(_rank_rows(x), _rank_rows(y))

### Phase 2 — Shared helpers and per-case metric functions

These functions are identical to those in `metrics_computation.ipynb`.

In [14]:
# ── Shared helpers (identical to metrics_computation.ipynb) ──

def _to_valid(x, y):
    x = np.asarray(x, dtype=float).ravel()
    y = np.asarray(y, dtype=float).ravel()
    valid = np.isfinite(x) & np.isfinite(y)
    return x[valid], y[valid]


def _safe_div(a, b):
    if b == 0 or not np.isfinite(b):
        return np.nan
    return a / b


def _minmax01(v):
    v = np.asarray(v, dtype=float)
    lo, hi = np.nanmin(v), np.nanmax(v)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return np.full_like(v, np.nan, dtype=float)
    return (v - lo) / (hi - lo)


def _endpoint_slope(x, y):
    if len(x) < 2:
        return np.nan
    return _safe_div(float(y[-1] - y[0]), float(x[-1] - x[0]))


def _polyfit_slope(x, y):
    if len(x) < 3 or np.std(x) == 0:
        return np.nan
    return float(np.polyfit(x, y, 1)[0])


def _segment_masks(n):
    i1, i2 = n // 3, 2 * n // 3
    early  = np.zeros(n, bool); early[:i1]   = True
    middle = np.zeros(n, bool); middle[i1:i2] = True
    late   = np.zeros(n, bool); late[i2:]    = True
    return early, middle, late


def _residual_sd(y, y_hat):
    r = y - y_hat
    return float(np.nanstd(r, ddof=1)) if len(r) >= 2 else np.nan


def _r2(y, y_hat):
    ss_res = float(np.nansum((y - y_hat) ** 2))
    ss_tot = float(np.nansum((y - np.nanmean(y)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan


def _sign_changes(x_curve, y_curve, tol=1e-6):
    x_curve = np.asarray(x_curve, float)
    y_curve = np.asarray(y_curve, float)
    valid = np.isfinite(x_curve) & np.isfinite(y_curve)
    x_curve, y_curve = x_curve[valid], y_curve[valid]
    if len(x_curve) < 4:
        return np.nan
    order = np.argsort(x_curve)
    x_curve, y_curve = x_curve[order], y_curve[order]
    dx = np.diff(x_curve)
    dy = np.diff(y_curve)
    ok = dx != 0
    if ok.sum() < 3:
        return np.nan
    slopes = dy[ok] / dx[ok]
    signs = np.zeros_like(slopes, dtype=int)
    signs[slopes > tol] = 1
    signs[slopes < -tol] = -1
    nz = signs[signs != 0]
    return float(np.sum(nz[1:] != nz[:-1])) if len(nz) >= 2 else 0.0


def _make_bins(x, n_bins=10, bin_type="equal_width"):
    if bin_type == "equal_width":
        return np.asarray(
            pd.cut(x, bins=n_bins, labels=False, include_lowest=True, duplicates="drop"),
            dtype=float,
        )
    elif bin_type == "equal_count":
        return np.asarray(
            pd.qcut(x, q=n_bins, labels=False, duplicates="drop"),
            dtype=float,
        )
    raise ValueError(f"Unknown bin_type: {bin_type}")

In [15]:
# ── Distance (identical to metrics_computation.ipynb) ──

def _double_center(a):
    a = a.reshape(-1, 1)
    dist = squareform(pdist(a))
    return dist - dist.mean(axis=0, keepdims=True) - dist.mean(axis=1, keepdims=True) + dist.mean()


def _distance_metrics(x, y):
    r = {"distance_covariance": np.nan, "distance_correlation": np.nan}
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return r
    A = _double_center(x)
    B = _double_center(y)
    dcov2_xy = (A * B).mean()
    dcov_xy = float(np.sqrt(max(dcov2_xy, 0)))
    dcov_xx = float(np.sqrt(max((A * A).mean(), 0)))
    dcov_yy = float(np.sqrt(max((B * B).mean(), 0)))
    r["distance_covariance"] = dcov_xy
    r["distance_correlation"] = _safe_div(dcov_xy, np.sqrt(dcov_xx * dcov_yy))
    return r


# ── MINE (identical) ──

def _mine_metrics(x, y):
    empty = {"MIC": np.nan, "MAS": np.nan, "MEV": np.nan, "MCN": np.nan, "MIC_minus_r2": np.nan}
    if not HAS_MINEPY or len(x) < 5:
        return empty
    try:
        mine = MINE(alpha=0.6, c=15)
        mine.compute_score(x, y)
        mic = mine.mic()
        r = pearsonr(x, y)[0] if np.std(x) > 0 and np.std(y) > 0 else np.nan
        return {
            "MIC": mic, "MAS": mine.mas(), "MEV": mine.mev(), "MCN": mine.mcn(),
            "MIC_minus_r2": mic - r ** 2 if np.isfinite(r) else np.nan,
        }
    except Exception:
        return empty


# ── Slopes (identical) ──

def _slope_metrics(x, y, prefix="raw"):
    """Endpoint + polyfit slopes for overall & 3 segments, plus segment strength."""
    keys = []
    for method in ["endpoint", "polyfit"]:
        for seg in ["overall", "early", "middle", "late"]:
            keys.append(f"{prefix}_{method}_{seg}_slope")
    keys.append(f"{prefix}_segment_strength")
    empty = {k: np.nan for k in keys}

    if len(x) < 6:
        return empty

    order = np.argsort(x)
    xs, ys = x[order], y[order]
    em, mm, lm = _segment_masks(len(xs))
    segments = {
        "overall": (xs, ys),
        "early":   (xs[em], ys[em]),
        "middle":  (xs[mm], ys[mm]),
        "late":    (xs[lm], ys[lm]),
    }

    r = {}
    ep_seg_abs = []
    for seg_name, (sx, sy) in segments.items():
        ep = _endpoint_slope(sx, sy)
        pf = _polyfit_slope(sx, sy)
        r[f"{prefix}_endpoint_{seg_name}_slope"] = ep
        r[f"{prefix}_polyfit_{seg_name}_slope"] = pf
        if seg_name != "overall":
            ep_seg_abs.append(abs(ep) if np.isfinite(ep) else np.nan)

    r[f"{prefix}_segment_strength"] = float(np.nanmean(ep_seg_abs))
    return r


def _standardized_slope_metrics(x, y):
    xn = _minmax01(x)
    yn = _minmax01(y)
    if np.any(np.isnan(xn)) or np.any(np.isnan(yn)):
        keys = []
        for method in ["endpoint", "polyfit"]:
            for seg in ["overall", "early", "middle", "late"]:
                keys.append(f"standardized_{method}_{seg}_slope")
        keys.append("standardized_segment_strength")
        return {k: np.nan for k in keys}
    return _slope_metrics(xn, yn, prefix="standardized")


# ── Bin metrics (identical) ──

def _bin_metrics(x, y, n_bins=10, bin_type="equal_width", min_count=5):
    """Amplitude, eta², buffer widths (q95−q05), n_valid_bins."""
    prefix = f"{bin_type}_bin"
    r = {
        f"{prefix}_amplitude": np.nan, f"{prefix}_eta_squared": np.nan,
        f"{prefix}_buffer_width_mean": np.nan,
        f"{prefix}_early_buffer_width": np.nan, f"{prefix}_middle_buffer_width": np.nan,
        f"{prefix}_late_buffer_width": np.nan, f"{prefix}_n_valid_bins": 0,
    }
    if len(x) < n_bins:
        return r
    try:
        bins = _make_bins(x, n_bins=n_bins, bin_type=bin_type)
    except Exception:
        return r
    df = pd.DataFrame({"x": x, "y": y, "bin": bins}).dropna()
    if df.empty:
        return r

    bin_stats = []
    for bid, g in df.groupby("bin", observed=True):
        if len(g) < min_count:
            continue
        yv = g["y"].values
        bw = float(np.nanpercentile(yv, 95) - np.nanpercentile(yv, 5))
        bin_stats.append({"bin": bid, "x_mean": float(g["x"].mean()),
                          "y_mean": float(yv.mean()), "buffer_width": bw, "count": len(g)})
    if len(bin_stats) < 2:
        return r

    bdf = pd.DataFrame(bin_stats).sort_values("x_mean").reset_index(drop=True)
    r[f"{prefix}_amplitude"] = float(bdf["y_mean"].max() - bdf["y_mean"].min())

    y_global = float(df["y"].mean())
    ss_tot = float(np.sum((df["y"].values - y_global) ** 2))
    ss_bet = sum(row["count"] * (row["y_mean"] - y_global) ** 2 for _, row in bdf.iterrows())
    r[f"{prefix}_eta_squared"] = _safe_div(ss_bet, ss_tot)

    r[f"{prefix}_buffer_width_mean"] = float(np.nanmean(bdf["buffer_width"]))
    r[f"{prefix}_n_valid_bins"] = len(bdf)
    nv = len(bdf)
    i1, i2 = nv // 3, 2 * nv // 3
    r[f"{prefix}_early_buffer_width"]  = float(np.nanmean(bdf.iloc[:i1]["buffer_width"]))
    r[f"{prefix}_middle_buffer_width"] = float(np.nanmean(bdf.iloc[i1:i2]["buffer_width"]))
    r[f"{prefix}_late_buffer_width"]   = float(np.nanmean(bdf.iloc[i2:]["buffer_width"]))
    return r


# ── X coverage (identical) ──

def _x_coverage_metrics(x, n_bins=10):
    r = {"x_bin_count_cv": np.nan, "x_uniform_ks_distance": np.nan}
    xf = x[np.isfinite(x)]
    if len(xf) < 3:
        return r
    lo, hi = xf.min(), xf.max()
    if hi > lo:
        xn = np.sort((xf - lo) / (hi - lo))
        n = len(xn)
        r["x_uniform_ks_distance"] = float(max(
            np.max(np.arange(1, n + 1) / n - xn),
            np.max(xn - np.arange(0, n) / n),
        ))
    if len(xf) >= n_bins:
        counts, _ = np.histogram(xf, bins=n_bins)
        mu = counts.mean()
        if mu > 0:
            r["x_bin_count_cv"] = float(np.std(counts, ddof=1) / mu)
    return r


# ── LOWESS (identical) ──

def _lowess_metrics(x, y, frac=0.25):
    empty = {
        "lowess_residual_sd": np.nan, "lowess_curve_amplitude": np.nan,
        "lowess_r2": np.nan, "lowess_first_derivative_sign_changes": np.nan,
        "lowess_overall_slope": np.nan,
    }
    if len(x) < 5:
        return empty
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    fitted = lowess(ys, xs, frac=frac, return_sorted=True)
    x_fit, y_fit = fitted[:, 0], fitted[:, 1]
    y_pred = np.interp(xs, x_fit, y_fit)
    return {
        "lowess_residual_sd": _residual_sd(ys, y_pred),
        "lowess_curve_amplitude": float(np.nanmax(y_fit) - np.nanmin(y_fit)),
        "lowess_r2": _r2(ys, y_pred),
        "lowess_first_derivative_sign_changes": _sign_changes(x_fit, y_fit),
        "lowess_overall_slope": _endpoint_slope(x_fit, y_fit),
    }

### Full-only metric groups

In [16]:
# ── Covariance ──

def _correlation_extra(x, y):
    r = {"covariance": np.nan}
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return r
    r["covariance"] = float(np.cov(x, y, ddof=1)[0, 1])
    return r


# ── Distribution (KS + Wasserstein) ──

def _distribution_metrics(x, y, n_bins=10, bin_type="equal_width"):
    prefix = f"{bin_type}_distribution"
    r = {f"{prefix}_ks_distance": np.nan, f"{prefix}_wasserstein_distance": np.nan}
    if len(x) < n_bins:
        return r
    try:
        bins = _make_bins(x, n_bins=n_bins, bin_type=bin_type)
    except Exception:
        return r
    df = pd.DataFrame({"x": x, "y": y, "bin": bins}).dropna()
    if df.empty:
        return r
    valid_bins = np.sort(df["bin"].unique())
    if len(valid_bins) < 3:
        return r
    nv = len(valid_bins)
    y_low = df[df["bin"].isin(valid_bins[: nv // 3])]["y"].values
    y_high = df[df["bin"].isin(valid_bins[2 * nv // 3 :])]["y"].values
    if len(y_low) < 2 or len(y_high) < 2:
        return r
    r[f"{prefix}_ks_distance"] = float(ks_2samp(y_low, y_high).statistic)
    r[f"{prefix}_wasserstein_distance"] = float(wasserstein_distance(y_low, y_high))
    return r


# ── GAM ──

def _gam_metrics(x, y, n_splines=10, lam=0.6, grid_size=200):
    empty = {
        "gam_residual_sd": np.nan, "gam_curve_amplitude": np.nan,
        "gam_r2": np.nan, "gam_first_derivative_sign_changes": np.nan,
        "gam_overall_slope": np.nan,
    }
    if not HAS_PYGAM or len(x) < 10:
        return empty
    try:
        gam = LinearGAM(gam_s(0, n_splines=n_splines), lam=lam).fit(x.reshape(-1, 1), y)
        x_curve = np.linspace(x.min(), x.max(), grid_size)
        y_curve = gam.predict(x_curve.reshape(-1, 1))
        y_pred = np.interp(x, x_curve, y_curve)
        return {
            "gam_residual_sd": _residual_sd(y, y_pred),
            "gam_curve_amplitude": float(np.nanmax(y_curve) - np.nanmin(y_curve)),
            "gam_r2": _r2(y, y_pred),
            "gam_first_derivative_sign_changes": _sign_changes(x_curve, y_curve),
            "gam_overall_slope": _endpoint_slope(x_curve, y_curve),
        }
    except Exception:
        return empty


# ── y-SD normalisation ──

_Y_SCALE_METRICS = [
    "equal_width_distribution_wasserstein_distance",
    "equal_count_distribution_wasserstein_distance",
    "equal_width_bin_amplitude", "equal_width_bin_buffer_width_mean",
    "equal_width_bin_early_buffer_width", "equal_width_bin_middle_buffer_width",
    "equal_width_bin_late_buffer_width",
    "equal_count_bin_amplitude", "equal_count_bin_buffer_width_mean",
    "equal_count_bin_early_buffer_width", "equal_count_bin_middle_buffer_width",
    "equal_count_bin_late_buffer_width",
    "lowess_residual_sd", "lowess_curve_amplitude",
    "gam_residual_sd", "gam_curve_amplitude",
]


def _ysd_normalized(y, metrics):
    y_sd = float(np.nanstd(y, ddof=1))
    r = {"y_sd": y_sd}
    for m in _Y_SCALE_METRICS:
        if m in metrics:
            r[f"{m}_div_y_sd"] = _safe_div(metrics[m], y_sd)
    return r

### Combined per-case wrapper

In [17]:
def compute_all_per_case(x: np.ndarray, y: np.ndarray) -> dict[str, float]:
    x, y = _to_valid(x, y)
    m: dict[str, float] = {}

    m.update(_correlation_extra(x, y))
    m.update(_distance_metrics(x, y))
    m.update(_x_coverage_metrics(x))

    m.update(_distribution_metrics(x, y, bin_type="equal_width"))
    m.update(_distribution_metrics(x, y, bin_type="equal_count"))

    m.update(_slope_metrics(x, y, prefix="raw"))
    m.update(_standardized_slope_metrics(x, y))
    m.update(_mine_metrics(x, y))

    m.update(_bin_metrics(x, y, bin_type="equal_width"))
    m.update(_bin_metrics(x, y, bin_type="equal_count"))

    m.update(_lowess_metrics(x, y))
    m.update(_gam_metrics(x, y))

    m.update(_ysd_normalized(y, m))
    m["n_valid"] = len(x)
    return m

### Quick single-case timing test

In [18]:
x_test = x_all[0].astype(np.float64)
y_test = y_all[0].astype(np.float64)

t0 = time.time()
sample = compute_all_per_case(x_test, y_test)
dt = time.time() - t0

print(f"Single case: {dt*1000:.1f} ms, {len(sample)} metrics")
print(f"Estimated total for {n_cases:,} cases: {dt * n_cases / 60:.0f} min")
print()
for k, v in sample.items():
    if isinstance(v, float) and np.isfinite(v):
        print(f"  {k:55s} = {v:.4f}")
    else:
        print(f"  {k:55s} = {v}")

Single case: 44.6 ms, 74 metrics
Estimated total for 114,176 cases: 85 min

  covariance                                              = 0.0072
  distance_covariance                                     = 0.0285
  distance_correlation                                    = 0.2580
  x_bin_count_cv                                          = 0.0909
  x_uniform_ks_distance                                   = 0.0195
  equal_width_distribution_ks_distance                    = 0.2780
  equal_width_distribution_wasserstein_distance           = 0.0570
  equal_count_distribution_ks_distance                    = 0.2750
  equal_count_distribution_wasserstein_distance           = 0.0569
  raw_endpoint_overall_slope                              = 0.0781
  raw_polyfit_overall_slope                               = 0.0873
  raw_endpoint_early_slope                                = -0.0741
  raw_polyfit_early_slope                                 = -0.0975
  raw_endpoint_middle_slope                        

## Compute All Metrics

In [19]:
# Phase 1: vectorised Pearson + Spearman
t0 = time.time()
x64 = x_all.astype(np.float64)
y64 = y_all.astype(np.float64)
pearson_all = vectorised_pearson(x64, y64)
spearman_all = vectorised_spearman(x64, y64)
del x64, y64
print(f"Phase 1 done: Pearson + Spearman for {n_cases:,} cases in {time.time() - t0:.1f}s")

/var/folders/y6/36h2pwl51ql85zwkh35jmm1r0000gp/T/ipykernel_75048/627309061.py:7: RuntimeWarning: invalid value encountered in divide
  return np.where(den > 0, num / den, np.nan)


Phase 1 done: Pearson + Spearman for 114,176 cases in 6.5s


In [20]:
# Phase 2: per-case metrics
REPORT_INTERVAL = 10_000

per_case_results: list[dict[str, float]] = []
t0 = time.time()

for i in tqdm(range(n_cases), desc="Per-case metrics"):
    x = x_all[i].astype(np.float64)
    y = y_all[i].astype(np.float64)
    per_case_results.append(compute_all_per_case(x, y))

    if (i + 1) % REPORT_INTERVAL == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        eta = (n_cases - i - 1) / rate
        print(f"  {i+1:>7,}/{n_cases:,}  ({rate:.0f} cases/s, ETA {eta/60:.1f} min)")

elapsed = time.time() - t0
print(f"\nPhase 2 done: {n_cases:,} cases in {elapsed/60:.1f} min ({n_cases / elapsed:.0f} cases/s)")

Per-case metrics:   9%|▉         | 10003/114176 [06:02<1:02:30, 27.78it/s]

   10,000/114,176  (28 cases/s, ETA 62.8 min)


Per-case metrics:  18%|█▊        | 20005/114176 [12:05<59:13, 26.50it/s]  

   20,000/114,176  (28 cases/s, ETA 56.9 min)


Per-case metrics:  26%|██▋       | 30004/114176 [18:06<54:31, 25.73it/s]  

...

  100,000/114,176  (27 cases/s, ETA 8.7 min)


Per-case metrics:  96%|█████████▋| 110001/114176 [1:12:45<05:13, 13.31it/s]

  110,000/114,176  (25 cases/s, ETA 2.8 min)


Per-case metrics: 100%|██████████| 114176/114176 [1:27:42<00:00, 21.70it/s]     


Phase 2 done: 114,176 cases in 87.7 min (22 cases/s)


## Save Results

In [21]:
metrics_df = pd.DataFrame(per_case_results)
metrics_df.insert(0, "case_id", cases_df["case_id"].values)
metrics_df["pearson_r"] = pearson_all
metrics_df["spearman_rho"] = spearman_all

col_order = [
    "case_id",
    # correlation
    "pearson_r", "spearman_rho", "covariance",
    # distance
    "distance_covariance", "distance_correlation",
    # x coverage
    "x_uniform_ks_distance", "x_bin_count_cv",
    # distribution
    "equal_width_distribution_ks_distance", "equal_width_distribution_wasserstein_distance",
    "equal_count_distribution_ks_distance", "equal_count_distribution_wasserstein_distance",
    # slopes — raw
    "raw_endpoint_overall_slope", "raw_endpoint_early_slope",
    "raw_endpoint_middle_slope", "raw_endpoint_late_slope",
    "raw_polyfit_overall_slope", "raw_polyfit_early_slope",
    "raw_polyfit_middle_slope", "raw_polyfit_late_slope",
    "raw_segment_strength",
    # slopes — standardized
    "standardized_endpoint_overall_slope", "standardized_endpoint_early_slope",
    "standardized_endpoint_middle_slope", "standardized_endpoint_late_slope",
    "standardized_polyfit_overall_slope", "standardized_polyfit_early_slope",
    "standardized_polyfit_middle_slope", "standardized_polyfit_late_slope",
    "standardized_segment_strength",
    # MINE
    "MIC", "MAS", "MEV", "MCN", "MIC_minus_r2",
    # bins equal-width
    "equal_width_bin_amplitude", "equal_width_bin_eta_squared",
    "equal_width_bin_buffer_width_mean",
    "equal_width_bin_early_buffer_width", "equal_width_bin_middle_buffer_width",
    "equal_width_bin_late_buffer_width", "equal_width_bin_n_valid_bins",
    # bins equal-count
    "equal_count_bin_amplitude", "equal_count_bin_eta_squared",
    "equal_count_bin_buffer_width_mean",
    "equal_count_bin_early_buffer_width", "equal_count_bin_middle_buffer_width",
    "equal_count_bin_late_buffer_width", "equal_count_bin_n_valid_bins",
    # lowess
    "lowess_residual_sd", "lowess_curve_amplitude", "lowess_r2",
    "lowess_first_derivative_sign_changes", "lowess_overall_slope",
    # gam
    "gam_residual_sd", "gam_curve_amplitude", "gam_r2",
    "gam_first_derivative_sign_changes", "gam_overall_slope",
    # y-sd normalised
    "y_sd",
    "equal_width_distribution_wasserstein_distance_div_y_sd",
    "equal_count_distribution_wasserstein_distance_div_y_sd",
    "equal_width_bin_amplitude_div_y_sd",
    "equal_width_bin_buffer_width_mean_div_y_sd",
    "equal_width_bin_early_buffer_width_div_y_sd",
    "equal_width_bin_middle_buffer_width_div_y_sd",
    "equal_width_bin_late_buffer_width_div_y_sd",
    "equal_count_bin_amplitude_div_y_sd",
    "equal_count_bin_buffer_width_mean_div_y_sd",
    "equal_count_bin_early_buffer_width_div_y_sd",
    "equal_count_bin_middle_buffer_width_div_y_sd",
    "equal_count_bin_late_buffer_width_div_y_sd",
    "lowess_residual_sd_div_y_sd", "lowess_curve_amplitude_div_y_sd",
    "gam_residual_sd_div_y_sd", "gam_curve_amplitude_div_y_sd",
    # misc
    "n_valid",
]

existing = [c for c in col_order if c in metrics_df.columns]
extra = [c for c in metrics_df.columns if c not in col_order]
metrics_df = metrics_df[existing + extra]

out_dir = DATA_DIR / "full"
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "metrics_full.parquet"
metrics_df.to_parquet(out_path, index=False)
size_mb = out_path.stat().st_size / 1e6
print(f"Saved {out_path}  ({len(metrics_df):,} rows × {len(metrics_df.columns)} cols, {size_mb:.1f} MB)")
print()
display(metrics_df.head())
print()
display(metrics_df.describe().T)

Saved generated_scatterplot_data/full/metrics_full.parquet  (114,176 rows × 77 cols, 77.9 MB)



,case_id,pearson_r,spearman_rho,covariance,distance_covariance,distance_correlation,x_uniform_ks_distance,x_bin_count_cv,equal_width_distribution_ks_distance,equal_width_distribution_wasserstein_distance,...,equal_count_bin_amplitude_div_y_sd,equal_count_bin_buffer_width_mean_div_y_sd,equal_count_bin_early_buffer_width_div_y_sd,equal_count_bin_middle_buffer_width_div_y_sd,equal_count_bin_late_buffer_width_div_y_sd,lowess_residual_sd_div_y_sd,lowess_curve_amplitude_div_y_sd,gam_residual_sd_div_y_sd,gam_curve_amplitude_div_y_sd,n_valid
0,1,0.270654,0.262035,0.007172,0.028464,0.257996,0.019547,0.090921,0.278046,0.057002,...,1.066362,2.790091,2.817159,2.779930,2.777411,0.956097,1.046995,0.954237,1.129189,500
1,2,0.202792,0.197157,0.002955,0.014718,0.186921,0.278087,0.674158,0.148052,0.035705,...,0.795384,3.072063,3.135177,3.103978,3.000792,0.966327,1.649095,0.968259,1.838513,500
2,3,0.141068,0.133403,0.002000,0.011786,0.151419,0.302452,0.759942,0.310241,0.043189,...,0.434430,3.103039,3.176726,3.207752,2.969239,0.985089,0.647537,0.986928,0.637510,500
3,4,0.188816,0.161542,0.002833,0.013518,0.170316,0.158937,0.605383,0.169806,0.045223,...,0.908254,3.067782,3.078500,3.075844,3.053697,0.966457,1.107108,0.971785,0.997866,500
4,5,0.208436,0.200662,0.004319,0.019181,0.187598,0.174817,1.084312,0.217091,0.047221,...,0.909747,3.075825,2.825160,3.250289,3.132975,0.968737,1.231108,0.975926,1.203576,500


,count,mean,std,min,25%,50%,75%,max
case_id,114176.0,57088.500000,32959.916505,1.000000e+00,28544.750000,57088.500000,85632.250000,114176.000000
pearson_r,114169.0,-0.000032,0.632792,-1.000000e+00,-0.615107,-0.002783,0.614752,1.000000
spearman_rho,114176.0,0.000070,0.612336,-1.000000e+00,-0.578733,-0.002734,0.580448,1.000000
covariance,114169.0,-0.001447,145.199786,-4.654889e+03,-0.044660,-0.000107,0.044765,5975.608368
distance_covariance,114169.0,0.309840,1.590472,3.475248e-04,0.073021,0.126870,0.202710,37.633561
...,...,...,...,...,...,...,...,...
lowess_residual_sd_div_y_sd,114169.0,0.578452,0.310149,0.000000e+00,0.301394,0.599988,0.878818,2.392956
lowess_curve_amplitude_div_y_sd,114169.0,2.801779,1.482963,0.000000e+00,1.858477,2.453878,3.710473,22.360680
gam_residual_sd_div_y_sd,114169.0,0.577977,0.302554,1.327781e-09,0.312023,0.601544,0.869617,0.999569
gam_curve_amplitude_div_y_sd,114169.0,2.956850,1.612542,1.049408e-01,1.928198,2.614797,3.737547,17.700663


## Sanity Checks

In [22]:
check = metrics_df.merge(cases_df[["case_id", "family_id", "snr", "spread_pattern", "x_distribution"]], on="case_id")

core = ["pearson_r", "spearman_rho", "distance_correlation", "MIC",
        "equal_width_bin_eta_squared", "lowess_r2", "gam_r2"]

null_const = (check["family_id"] == "Null") & (check["spread_pattern"] == "constant")
print("=== Null (constant spread) — expect near 0 ===")
for col in core:
    if col not in check.columns:
        continue
    vals = check.loc[null_const, col].dropna()
    if len(vals):
        print(f"  {col:35s}  mean={vals.mean():+.4f}  |mean|={vals.abs().mean():.4f}")
print()

strong = ((check["family_id"] == "F01") & (check["snr"].astype(str) == "100")
          & (check["spread_pattern"] == "constant") & (check["x_distribution"] == "even"))
print("=== F01 Linear, SNR=100 — expect high ===")
for col in core:
    if col not in check.columns:
        continue
    vals = check.loc[strong, col].dropna()
    if len(vals):
        print(f"  {col:35s}  mean={vals.mean():.4f}")
print()

ushape = ((check["family_id"] == "F18") & (check["snr"].astype(str) == "100")
          & (check["spread_pattern"] == "constant") & (check["x_distribution"] == "even"))
print("=== F18 U-shape, SNR=100 — |Pearson|≈0, dcor/MIC high ===")
for col in core:
    if col not in check.columns:
        continue
    vals = check.loc[ushape, col].dropna()
    if len(vals):
        print(f"  {col:35s}  mean={vals.mean():+.4f}  |mean|={vals.abs().mean():.4f}")

print(f"\nTotal columns: {len(metrics_df.columns)}")

=== Null (constant spread) — expect near 0 ===
  pearson_r                            mean=-0.0084  |mean|=0.0316
  spearman_rho                         mean=-0.0064  |mean|=0.0292
  distance_correlation                 mean=+0.0767  |mean|=0.0767
  MIC                                  mean=+0.1635  |mean|=0.1635
  equal_width_bin_eta_squared          mean=+0.0180  |mean|=0.0180
  lowess_r2                            mean=+0.0033  |mean|=0.0202
  gam_r2                               mean=+0.0152  |mean|=0.0152

=== F01 Linear, SNR=100 — expect high ===

=== F18 U-shape, SNR=100 — |Pearson|≈0, dcor/MIC high ===

Total columns: 77
